# main matrix

**Purpose**: run the full main paper experiment matrix: Random-K=2, KATE, JAC, CIDER across 3-4 backbones × up to 8 pairs × 3 seeds.

**Prerequisite**: run CIDER baseline probe cells 1-19 first for SBERT + structural + selection functions to be loaded.


## 1. Bootstrap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
BASE = os.environ.get('REPO_ROOT') or '/content/drive/MyDrive/cd-er-paradigm-choice'
os.chdir(BASE)
os.environ['REPO_ROOT'] = BASE
sys.path.insert(0, BASE)

In [ ]:
%pip install -q sentence-transformers scikit-learn python-Levenshtein jellyfish together openai

## 2. Config — the matrix

Set `SUBSET` to control which cells run:
- `"phase1"`: Random + CIDER on 3 backbones × 4 clean pairs × 3 seeds
- `"phase2"`: KATE + JAC on same matrix
- `"phase3"`: All 5 methods on 4 new pairs
- `"phase5"`: GPT-4o-mini subset (optional)
- `"all"`: full matrix (all phases)

Each phase writes to `results/runs/main_matrix/<backbone>/<pair>/<method>/seed_<n>/metrics.json`.

In [ ]:
# ---------------------------------------------------------------------------
# SELECT WHICH SUBSET TO RUN
# ---------------------------------------------------------------------------
SUBSET = "phase_p5_topup"    # "phase1" | "phase2" | "phase3" | "phase5" | "all" | "phase_zs"

# ---------------------------------------------------------------------------
# Backbones
# ---------------------------------------------------------------------------
# Provider switched from Together AI to DeepInfra: Together AI restricted
# 8B and 72B models on our account tier to dedicated endpoints only, but
# DeepInfra hosts all three on serverless at lower cost per token.
BACKBONES = {
    "llama-3.3-70b": {
        "provider": "deepinfra",
        "model": "meta-llama/Llama-3.3-70B-Instruct",
    },
    "llama-3.1-8b": {
        "provider": "deepinfra",
        "model": "meta-llama/Meta-Llama-3.1-8B-Instruct",
    },
    "qwen-2.5-72b": {
        "provider": "deepinfra",
        "model": "Qwen/Qwen2.5-72B-Instruct",
    },
    # Uncomment for Phase 5
    # "gpt-4o-mini": {"provider": "openai", "model": "gpt-4o-mini"},
}

# ---------------------------------------------------------------------------
# Pairs (source, target)
# ---------------------------------------------------------------------------
CLEAN_PAIRS = [
    ("Structured/Walmart-Amazon", "Textual/Abt-Buy"),        # P1
    ("wdc/computers",              "wdc/watches"),           # P2
    ("Structured/Walmart-Amazon", "Structured/DBLP-ACM"),    # P3
    ("Structured/Walmart-Amazon", "Structured/Amazon-Google"), # P4
]
NEW_PAIRS = [
    # ("Structured/Walmart-Amazon", "Dirty/DBLP-ACM"),         # P5: done in the baseline probe
    ("Structured/Walmart-Amazon", "Dirty/Walmart-Amazon"),     # P6 dirty target
    ("Structured/iTunes-Amazon",  "Structured/DBLP-ACM"),      # P7 music -> citation
    ("Structured/Walmart-Amazon", "Structured/DBLP-Scholar"),  # P8 additional citation
    # add more as needed
]

# ---------------------------------------------------------------------------
# Methods
# ---------------------------------------------------------------------------
METHODS = ["random_k2", "kate_k2", "jac_k2", "cider_k2"]
# ZS is not run per-seed (deterministic); tracked separately

# ---------------------------------------------------------------------------
# Seeds
# ---------------------------------------------------------------------------
# SEEDS = [42, 123, 456]
SEEDS = [42]

# ---------------------------------------------------------------------------
# CIDER hparams (using earlier calibrated values; can be tuned per backbone)
# ---------------------------------------------------------------------------
GAMMA = 0.01
ALPHA = 0.5
H_CANDIDATES = 50
K_DEMOS = 2
KFOLDS = 5

SKIP_IF_DONE = True

# Determine which cells to run based on SUBSET
def get_run_config():
    if SUBSET == "phase1":
        return {"backbones": ["llama-3.3-70b", "llama-3.1-8b", "qwen-2.5-72b"],
                "pairs": CLEAN_PAIRS, "methods": ["random_k2", "cider_k2"]}
    if SUBSET == "phase2":
        return {"backbones": ["llama-3.3-70b", "llama-3.1-8b", "qwen-2.5-72b"],
                "pairs": CLEAN_PAIRS, "methods": ["kate_k2", "jac_k2"]}
    if SUBSET == "phase3":
        return {"backbones": ["llama-3.3-70b", "llama-3.1-8b", "qwen-2.5-72b"],
                "pairs": NEW_PAIRS, "methods": METHODS}
    if SUBSET == "phase5":
        return {"backbones": ["gpt-4o-mini"], "pairs": CLEAN_PAIRS,
                "methods": ["random_k2", "cider_k2"]}
    if SUBSET == "all":
        return {"backbones": list(BACKBONES.keys()),
                "pairs": CLEAN_PAIRS + NEW_PAIRS, "methods": METHODS}
    if SUBSET == "phase_zs":
        return {"backbones": ["llama-3.3-70b", "llama-3.1-8b", "qwen-2.5-72b"],
                "pairs": CLEAN_PAIRS, "methods": ["zero_shot"]}
    if SUBSET == "phase_p5_topup":
      return {"backbones": ["llama-3.3-70b"],
              "pairs": [("Structured/Walmart-Amazon", "Dirty/DBLP-ACM")],
              "methods": ["random_k2"]}

    raise ValueError(f"Unknown SUBSET: {SUBSET}")

RUN_CONFIG = get_run_config()
print(f"Running SUBSET={SUBSET}")
print(f"  backbones: {RUN_CONFIG['backbones']}")
print(f"  pairs: {len(RUN_CONFIG['pairs'])} pairs")
print(f"  methods: {RUN_CONFIG['methods']}")
print(f"  seeds: {SEEDS}")
print(f"  total cells: {len(RUN_CONFIG['backbones']) * len(RUN_CONFIG['pairs']) * len(RUN_CONFIG['methods']) * len(SEEDS)}")

## 3. API clients (for all backbones)

In [ ]:
from google.colab import userdata
from openai import OpenAI

# Provider-to-endpoint mapping. Each provider is OpenAI-SDK-compatible with a
# base_url swap and an API key. Add or remove entries as needed.
PROVIDER_ENDPOINTS = {
    "together":  ("TOGETHER_API_KEY",  "https://api.together.xyz/v1"),
    "deepinfra": ("DEEPINFRA_API_KEY", "https://api.deepinfra.com/v1/openai"),
    "fireworks": ("FIREWORKS_API_KEY", "https://api.fireworks.ai/inference/v1"),
    "openai":    ("OPENAI_API_KEY",    None),   # None = SDK default
}


def make_client(backbone_config):
    """Return (client, model_name) for a backbone spec."""
    provider = backbone_config["provider"]
    if provider not in PROVIDER_ENDPOINTS:
        raise ValueError(f"Unknown provider: {provider}. Add it to PROVIDER_ENDPOINTS.")
    secret_key, base_url = PROVIDER_ENDPOINTS[provider]
    api_key = userdata.get(secret_key)
    if not api_key:
        raise RuntimeError(
            f"Missing Colab secret {secret_key!r} for provider {provider!r}. "
            f"Add it in Colab (left sidebar → key icon) and enable notebook access."
        )
    kwargs = {"api_key": api_key}
    if base_url is not None:
        kwargs["base_url"] = base_url
    return OpenAI(**kwargs), backbone_config["model"]

# Precreate clients (they're cheap)
CLIENTS = {name: make_client(cfg) for name, cfg in BACKBONES.items()
           if name in RUN_CONFIG["backbones"]}
for name, (client, model) in CLIENTS.items():
    print(f"  {name}: {model}")

## 4. New selection functions (KATE, JAC, Random-stratified)

Add these to the earlier baseline's function set. Each returns a `(n_target, k)` array of candidate indices.

In [ ]:
import numpy as np
import random as _rand


def select_kate_demos(target_sem, source_sem, k=2, **_):
    """Top-K source by SBERT cosine similarity to each target. No filter, no stratification."""
    def _cos(a, b):
        an = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-10)
        bn = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-10)
        return an @ bn.T
    sim = _cos(target_sem, source_sem)
    return np.argsort(-sim, axis=1)[:, :k]


def select_jac_demos(target_pairs, source_pairs, k=2, **_):
    """Top-K source by Jaccard on serialized concatenation of left+right. No filter, no stratification."""
    def _serialize(p):
        return f"{p['left']} {p['right']}"
    def _jaccard_tokens(s1, s2):
        t1, t2 = set(s1.lower().split()), set(s2.lower().split())
        if not (t1 or t2):
            return 0.0
        return len(t1 & t2) / len(t1 | t2)
    src_strs = [_serialize(p) for p in source_pairs]
    demos = np.zeros((len(target_pairs), k), dtype=np.int64)
    for i, tp in enumerate(target_pairs):
        ts = _serialize(tp)
        sims = np.array([_jaccard_tokens(ts, s) for s in src_strs])
        demos[i] = np.argsort(-sims)[:k]
    return demos


def select_random_stratified_demos(source_pairs, k=2, seed=42):
    """1 positive + 1 negative random demo, PER-TARGET randomised via seed. Fixed demos across all targets."""
    rng = _rand.Random(seed)
    pos = [p for p in source_pairs if p["label"] == 1]
    neg = [p for p in source_pairs if p["label"] == 0]
    n_pos, n_neg = k // 2, k - k // 2
    return (rng.sample(pos, min(n_pos, len(pos)))
            + rng.sample(neg, min(n_neg, len(neg))))

print("[ok] selectors: select_kate_demos, select_jac_demos, select_random_stratified_demos loaded")

## 5. Backbone-parameterised LLM call

In [ ]:
import time


def llm_call_with(client, model, prompt, max_retries=3, max_tokens=10):
    """Call the chat.completions endpoint and return usage/latency alongside text.

    Returns (text, prompt_tokens, completion_tokens, api_latency_sec).
    On persistent failure returns ("", 0, 0, 0.0)."""
    for attempt in range(max_retries):
        try:
            t0 = time.perf_counter()
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens,
                temperature=0.0,
            )
            latency = time.perf_counter() - t0
            usage = getattr(resp, "usage", None)
            prompt_tokens = getattr(usage, "prompt_tokens", 0) if usage else 0
            completion_tokens = getattr(usage, "completion_tokens", 0) if usage else 0
            return resp.choices[0].message.content.strip(), prompt_tokens, completion_tokens, latency
        except Exception as e:
            if attempt == max_retries - 1:
                return "", 0, 0, 0.0
            time.sleep(2 ** attempt)
    return "", 0, 0, 0.0


## 6. Domain info (extend for new pairs)

In [ ]:
DOMAIN_INFO_EXTENDED = {
    "Textual/Abt-Buy": "product",
    "Structured/Walmart-Amazon": "product",
    "Structured/Amazon-Google": "product",
    "Structured/DBLP-ACM": "citation",
    "Structured/DBLP-Scholar": "citation",
    "Dirty/DBLP-ACM": "citation",
    "Dirty/DBLP-Scholar": "citation",
    "Dirty/Walmart-Amazon": "product",
    "Dirty/iTunes-Amazon": "music",
    "Structured/iTunes-Amazon": "music",
    "wdc/computers": "product",
    "wdc/watches": "product",
    "wdc/cameras": "product",
    "wdc/shoes": "product",
    "Textual/Company": "company",
    "Structured/Beer": "beer product",
}
# Override the CIDER baseline probe dict
DOMAIN_INFO = DOMAIN_INFO_EXTENDED

## 6b. CIDER-helpers helpers (inlined for self-containment)

In [ ]:
# ================================================================
# CIDER pipeline helpers (inlined from CIDER baseline probe for self-containment)
# matrix-specific notes:
#   * SOURCE_FOR_TARGET is defined below so run_one_cell resolves target pairs
# ================================================================

# --- sys.path insurance so config.config resolves even after a kernel restart ---
import sys, os
_BASE = os.environ.get('REPO_ROOT') or '/content/drive/MyDrive/cd-er-paradigm-choice'
if _BASE not in sys.path:
    sys.path.insert(0, _BASE)
if os.path.isdir(_BASE) and os.getcwd() != _BASE:
    os.chdir(_BASE)
os.environ.setdefault('REPO_ROOT', _BASE)

# --- SOURCE_FOR_TARGET mapping (mirrors CIDER baseline) ---
SOURCE_FOR_TARGET = {
    "Textual/Abt-Buy":          "Structured/Walmart-Amazon",
    "wdc/watches":              "wdc/computers",
    "Structured/DBLP-ACM":      "Structured/Walmart-Amazon",
    "Structured/Amazon-Google": "Structured/Walmart-Amazon",
    "Dirty/DBLP-ACM":           "Structured/Walmart-Amazon",
    "Structured/DBLP-Scholar":  "Structured/Walmart-Amazon",
    "Dirty/Walmart-Amazon":     "Structured/Walmart-Amazon",
}

# --- TARGETS alias — the inlined CIDER baseline probe probe loop iterates TARGETS,
# but main matrix uses RUN_CONFIG["pairs"] as the source of truth. Alias
# TARGETS to CLEAN_PAIRS' target column so the probe can resolve.
TARGETS = [target for (_source, target) in CLEAN_PAIRS]

# --- SBERT model + CIDER defaults referenced by inlined helpers ---
SBERT_MODEL = "sentence-transformers/all-mpnet-base-v2"
PROVIDER = "deepinfra"   # informational; per-backbone provider is on cfg dict

# --- from CIDER baseline probe cell 9 ---
# %% Load splits, reuses MatchGPT sub-analysis logic
from pathlib import Path

def load_ditto_split(path):
    """Load tab-separated Ditto file: left\tright\tlabel per line."""
    pairs = []
    with open(path) as f:
        for line in f:
            parts = line.rstrip().split("\t")
            if len(parts) >= 3:
                pairs.append({"left": parts[0], "right": parts[1], "label": int(parts[2])})
    return pairs

def get_source_train_and_target_test(target_dataset):
    """Return (source_train_pairs, target_test_pairs, source_dataset)."""
    from config.config import PATHS
    source_dataset = SOURCE_FOR_TARGET[target_dataset]

    # Source train (handle WDC vs ER-Magellan)
    if source_dataset.startswith("wdc/"):
        cat = source_dataset.split("/")[-1]
        wdc_dir = PATHS.ditto_repo / "data" / "wdc" / cat
        source_train_path = None
        for suffix in (".large", ".xlarge", ".medium", ".small"):
            cand = wdc_dir / f"train.txt{suffix}"
            if cand.exists():
                source_train_path = cand
                break
        if source_train_path is None:
            raise FileNotFoundError(f"No train.txt.* in {wdc_dir}")
    else:
        source_train_path = PATHS.ditto_repo / "data" / "er_magellan" / source_dataset / "train.txt"

    # Target test (same logic)
    if target_dataset.startswith("wdc/"):
        cat = target_dataset.split("/")[-1]
        target_test_path = PATHS.ditto_repo / "data" / "wdc" / cat / "test.txt"
    else:
        target_test_path = PATHS.ditto_repo / "data" / "er_magellan" / target_dataset / "test.txt"

    return (load_ditto_split(source_train_path),
            load_ditto_split(target_test_path),
            source_dataset)

# Quick test
print("=== Probing data paths for the 4 target pairs ===")
for tgt in TARGETS:
    try:
        src_train, tgt_test, src = get_source_train_and_target_test(tgt)
        print(f"[ok] {src} -> {tgt}: source_train={len(src_train)}, target_test={len(tgt_test)}")
    except Exception as e:
        print(f"[ERR] {tgt}: {e}")

# --- from CIDER baseline probe cell 11 ---
# %% SBERT encoder
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# Load once, reuse across pairs
print(f"[load] {SBERT_MODEL}")
device = "cuda" if torch.cuda.is_available() else "cpu"
sbert = SentenceTransformer(SBERT_MODEL, device=device)
print(f"[ok] SBERT loaded on {device}, dim={sbert.get_sentence_embedding_dimension()}")

def encode_entity_pairs(pairs, batch_size=64):
    """Encode list of {left, right, label} dicts into (N, 768) float32 array.
    Concatenates left + [SEP] + right and passes through SBERT.
    """
    texts = [f"{p['left']} [SEP] {p['right']}" for p in pairs]
    return sbert.encode(texts, batch_size=batch_size, show_progress_bar=True,
                        convert_to_numpy=True, normalize_embeddings=False).astype(np.float32)

# --- from CIDER baseline probe cell ---
# %% Parse Ditto format + compute 3-d structural vector
import re
import Levenshtein
import jellyfish

DITTO_ATTR_RE = re.compile(r'COL\s+(\S+)\s+VAL\s+(.+?)(?=\s*COL\s+\S+\s+VAL|$)', re.DOTALL)

def parse_ditto_entity(entity_str):
    """Parse 'COL title VAL "..." COL brand VAL "..."' into {title: "...", brand: "..."}."""
    if 'COL' not in entity_str:
        # Fallback: not in Ditto COL/VAL format, treat as single 'text' attribute
        return {'text': entity_str.strip()}
    matches = DITTO_ATTR_RE.findall(entity_str)
    if not matches:
        return {'text': entity_str.strip()}
    return {name.strip(): val.strip() for name, val in matches}

def jaccard_tokens(s1, s2):
    t1 = set(s1.lower().split())
    t2 = set(s2.lower().split())
    if not (t1 or t2):
        return 0.0
    return len(t1 & t2) / len(t1 | t2)

def levenshtein_norm(s1, s2):
    if max(len(s1), len(s2)) == 0:
        return 0.0
    return 1.0 - Levenshtein.distance(s1, s2) / max(len(s1), len(s2))

def jaro_winkler_norm(s1, s2):
    return jellyfish.jaro_winkler_similarity(s1, s2)

def structural_vector_pair(left_str, right_str):
    """3-d vector: [Jaccard, Levenshtein, Jaro-Winkler] averaged across attributes
    shared by both entities within the pair. Domain-invariant (Eq. 7-8 of CIDER)."""
    left_attrs = parse_ditto_entity(left_str)
    right_attrs = parse_ditto_entity(right_str)
    common = set(left_attrs) & set(right_attrs)
    if not common:
        # Degenerate: no shared attributes → all zeros
        return np.zeros(3, dtype=np.float32)
    jaccs, levs, jaros = [], [], []
    for attr in common:
        l, r = str(left_attrs[attr]), str(right_attrs[attr])
        jaccs.append(jaccard_tokens(l, r))
        levs.append(levenshtein_norm(l, r))
        jaros.append(jaro_winkler_norm(l, r))
    return np.array([np.mean(jaccs), np.mean(levs), np.mean(jaros)], dtype=np.float32)

def structural_vectors_for_pairs(pairs):
    """Compute (N, 3) structural feature matrix."""
    return np.stack([structural_vector_pair(p['left'], p['right']) for p in pairs])

# Sanity check
sample = {'left': 'COL title VAL "canon powershot a3300" COL brand VAL "canon"',
          'right': 'COL title VAL "canon powershot elph 100" COL brand VAL "canon"'}
print(f"Sample structural vector: {structural_vector_pair(sample['left'], sample['right'])}")

# --- from CIDER baseline probe cell ---
# %% Active candidate source generation (Step 1)
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import KFold

def kfold_match_probabilities(X, y, k=KFOLDS):
    """For each source pair, return P(match | v) from K-fold Naive Bayes."""
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    probs = np.zeros(len(y), dtype=np.float64)
    for train_idx, val_idx in kf.split(X):
        nb = GaussianNB()
        nb.fit(X[train_idx], y[train_idx])
        probs[val_idx] = nb.predict_proba(X[val_idx])[:, 1]
    return probs

def binary_entropy(probs):
    """H = -p log2(p) - (1-p) log2(1-p), per Eq. 1."""
    p = np.clip(probs, 1e-10, 1 - 1e-10)
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)

def cider_value(uncertainties, src_vectors, tgt_mean_vector, gamma=GAMMA):
    """Eq. 6: value(a,b) = entropy(a,b) + gamma * 1 / dist(v, v*)"""
    dists = np.linalg.norm(src_vectors - tgt_mean_vector, axis=1)
    inv_dists = 1.0 / (dists + 1e-10)
    return uncertainties + gamma * inv_dists

def select_candidate_source(source_pairs, source_vectors, target_vectors,
                             h=H_CANDIDATES, gamma=GAMMA):
    """Step 1 of CIDER. Returns indices of top-h candidate source pairs."""
    y = np.array([p['label'] for p in source_pairs], dtype=np.int64)
    # Uncertainty
    probs = kfold_match_probabilities(source_vectors, y)
    entropies = binary_entropy(probs)
    # Mean target vector
    v_star = target_vectors.mean(axis=0)
    # Value (Eq. 6)
    values = cider_value(entropies, source_vectors, v_star, gamma=gamma)
    # Top-h
    top_idx = np.argsort(-values)[:h]
    return top_idx, values

# --- from CIDER baseline probe cell ---
# %% In-context demo selection (Step 2) — STRATIFIED variant
# Deviation from CIDER paper: the paper uses pure top-K by similarity (Eq. 9).
# On extreme cross-domain pairs (product source -> citation target), pure top-K
# collapses to all-negative demos because (a) active candidate selection picks
# uncertainty=1.0 source pairs which are mostly hard negatives and (b) cross-
# domain semantic similarity is near 0, so structural similarity dominates.
# This stratified variant picks top-(K/2) positives + top-(K/2) negatives by
# similarity to guarantee class balance in the prompt. Disclose this in §V.F.4.

def cosine_sim_rows(A, B):
    """Pairwise cosine similarity between rows of A (n, d) and rows of B (m, d).
    Returns (n, m)."""
    A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-10)
    B_norm = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-10)
    return A_norm @ B_norm.T

def cider_similarity_matrix(target_sem, candidate_sem, target_struct, candidate_struct,
                             alpha=ALPHA):
    """Eq. 9: sim = alpha * cos(sem) + (1-alpha) * cos(struct).
    Returns (n_target, n_candidate) matrix."""
    sem_sim = cosine_sim_rows(target_sem, candidate_sem)
    struct_sim = cosine_sim_rows(target_struct, candidate_struct)
    return alpha * sem_sim + (1 - alpha) * struct_sim

def select_demos_for_targets(sim_matrix, k=K_DEMOS, candidate_labels=None):
    """Stratified top-K demo selection. If candidate_labels is None, falls back
    to pure top-K (faithful to CIDER paper). If provided, picks top-(k/2)
    positives + top-(k/2) negatives by similarity per target row."""
    if candidate_labels is None:
        # Faithful CIDER: pure top-K by similarity
        return np.argsort(-sim_matrix, axis=1)[:, :k]

    # Stratified variant
    candidate_labels = np.asarray(candidate_labels)
    n_target, n_cand = sim_matrix.shape
    pos_idx = np.where(candidate_labels == 1)[0]
    neg_idx = np.where(candidate_labels == 0)[0]
    n_pos = k // 2
    n_neg = k - n_pos

    demos = np.zeros((n_target, k), dtype=np.int64)
    for i in range(n_target):
        # Top-n_pos positives by similarity
        if len(pos_idx) > 0:
            top_pos = pos_idx[np.argsort(-sim_matrix[i, pos_idx])[:n_pos]]
        else:
            top_pos = np.array([], dtype=np.int64)
        # Top-n_neg negatives by similarity
        if len(neg_idx) > 0:
            top_neg = neg_idx[np.argsort(-sim_matrix[i, neg_idx])[:n_neg]]
        else:
            top_neg = np.array([], dtype=np.int64)
        combined = np.concatenate([top_pos, top_neg])
        # Fallback: if one class was empty, fill remaining slots from any class
        if len(combined) < k:
            seen = set(combined.tolist())
            for cand_idx in np.argsort(-sim_matrix[i]):
                if cand_idx not in seen:
                    combined = np.append(combined, cand_idx)
                    seen.add(cand_idx)
                    if len(combined) == k:
                        break
        demos[i] = combined[:k]
    return demos

# --- from CIDER baseline probe cell ---
# %% Build CIDER prompt
def build_cider_prompt(target_pair, demo_pairs, source_domain, target_domain):
    """Per Fig. 3 of CIDER + explicit Yes/No instruction for LLaMA-3.3-70B
    (verbose-by-default; would otherwise produce explanatory text instead of
    a yes/no answer)."""
    lines = []
    src_cap = source_domain.capitalize()
    tgt_cap = target_domain.capitalize()
    for d in demo_pairs:
        lines.append(f"Do the two following {source_domain} descriptions refer to the same {source_domain}?")
        lines.append(f"{src_cap}1: {d['left']}")
        lines.append(f"{src_cap}2: {d['right']}")
        lines.append("Yes" if d['label'] == 1 else "No")
        lines.append("")
    lines.append(f"Do the two following {target_domain} descriptions refer to the same {target_domain}?")
    lines.append(f"{tgt_cap}1: {target_pair['left']}")
    lines.append(f"{tgt_cap}2: {target_pair['right']}")
    lines.append("Answer with Yes or No only.")   # required for LLaMA-3.3-70B
    return "\n".join(lines)

# Sanity check
demo_pair = {'left': 'COL title VAL canon a3300', 'right': 'COL title VAL canon elph 100', 'label': 0}
tgt_pair  = {'left': 'COL title VAL canon a3400', 'right': 'COL title VAL canon a3300', 'label': 1}
print(build_cider_prompt(tgt_pair, [demo_pair, demo_pair], "product", "product")[:700])

# --- from CIDER baseline probe cell (parse_yes_no only; main matrix uses its own llm_call_with) ---
def parse_yes_no(text):
    """Parse Yes/No (or Match/No match synonyms) from model response.
    Returns 1 for yes/match, 0 for no/not-match/empty/ambiguous."""
    t = text.lower().strip()
    # Priority 1: response starts with a clear answer token
    if t.startswith("yes") or t.startswith("match") or t.startswith("same"):
        return 1
    if t.startswith("no"):
        return 0
    # Priority 2: synonym phrases anywhere in response
    if "yes" in t and "no" not in t:
        return 1
    if ("no match" in t) or ("not the same" in t) or ("different" in t and "same" not in t):
        return 0
    if "match" in t and "no match" not in t:
        return 1
    if "no" in t and "yes" not in t:
        return 0
    return 0   # default: ambiguous → treat as not-match

# --- from CIDER baseline probe cell ---
# %% Compute F1 + save metrics
def compute_metrics(predictions, labels):
    tp = sum(1 for p, l in zip(predictions, labels) if p == 1 and l == 1)
    fp = sum(1 for p, l in zip(predictions, labels) if p == 1 and l == 0)
    fn = sum(1 for p, l in zip(predictions, labels) if p == 0 and l == 1)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return f1, precision, recall

def save_metrics(out_dir, payload):
    import json
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / "metrics.json", "w") as f:
        json.dump(payload, f, indent=2)
    print(f"[wrote] {out_dir / 'metrics.json'}")

print("[ok] CIDER-helpers helpers inlined (data loading, SBERT, CIDER pipeline, prompt, parse_yes_no, compute_metrics)")


## 7. Per-cell runner, one (backbone, pair, method, seed) → metrics.json

In [ ]:
import json
from pathlib import Path

OUT_BASE = Path(BASE) / "results" / "runs" / "main_matrix"


# ------------------------------------------------------------
# SBERT feature cache: encoded features per (source, target) pair
# are deterministic regardless of backbone / method / seed, so we
# cache them on disk and reuse across cells. Massive wall-clock
# savings on kate_k2 and cider_k2 cells; random_k2 and jac_k2 skip
# SBERT entirely.
# ------------------------------------------------------------
import numpy as np
_SBERT_CACHE = {}
_SBERT_CACHE_DIR = Path(BASE) / "results" / "cache" / "sbert"
_SBERT_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def _get_sbert_features(source_train, target_test, source_ds, target_ds):
    """Return (src_sem, tgt_sem) cached by (source_ds, target_ds).

    First checks in-memory dict, then disk cache under
    results/cache/sbert/, then encodes and persists both."""
    key = (source_ds, target_ds)
    if key in _SBERT_CACHE:
        return _SBERT_CACHE[key]
    cache_file = _SBERT_CACHE_DIR / (
        source_ds.replace("/", "__") + "__" + target_ds.replace("/", "__") + ".npz"
    )
    if cache_file.exists():
        d = np.load(cache_file)
        pair = (d["src_sem"], d["tgt_sem"])
        _SBERT_CACHE[key] = pair
        print(f"    [sbert] loaded cached features from {cache_file.name}")
        return pair
    print(f"    [sbert] encoding source ({len(source_train)}) + target ({len(target_test)})...")
    src_sem = encode_entity_pairs(source_train)
    tgt_sem = encode_entity_pairs(target_test)
    _SBERT_CACHE[key] = (src_sem, tgt_sem)
    np.savez(cache_file, src_sem=src_sem, tgt_sem=tgt_sem)
    print(f"    [sbert] cached to {cache_file.name}")
    return src_sem, tgt_sem


def run_one_cell(backbone_name, source_ds, target_ds, method, seed):
    """One matrix cell: backbone x pair x method x seed."""
    pair_tag = target_ds.replace("/", "__")
    out_dir = OUT_BASE / backbone_name / pair_tag / method / f"seed_{seed}"
    metrics_path = out_dir / "metrics.json"

    if SKIP_IF_DONE and metrics_path.exists():
        try:
            existing = json.load(open(metrics_path))
            if existing.get("test_f1") is not None and existing.get("returncode") == 0:
                print(f"  [skip] {backbone_name} / {pair_tag} / {method} / seed_{seed}: "
                      f"F1={existing['test_f1']:.4f}")
                return existing
        except Exception:
            pass

    # Load pair data (may need to override SOURCE_FOR_TARGET)
    global SOURCE_FOR_TARGET
    if target_ds not in SOURCE_FOR_TARGET:
        SOURCE_FOR_TARGET[target_ds] = source_ds

    source_train, target_test, resolved_source = get_source_train_and_target_test(target_ds)
    n_test = len(target_test)
    pos_rate = sum(p["label"] for p in target_test) / n_test
    src_dom = DOMAIN_INFO.get(source_ds, "entity")
    tgt_dom = DOMAIN_INFO.get(target_ds, "entity")

    t_start = time.time()
    print(f"\n=== {backbone_name} / {target_ds} / {method} / seed_{seed} ===")

    # SBERT encoding only needed by kate_k2 and cider_k2; cached by (source, target)
    if method in ("kate_k2", "cider_k2"):
        src_sem, tgt_sem = _get_sbert_features(source_train, target_test, resolved_source, target_ds)
    else:
        src_sem, tgt_sem = None, None

    # Method-specific demo selection
    if method == "random_k2":
        demos_fixed = select_random_stratified_demos(source_train, k=2, seed=seed)
        demo_getter = lambda i: demos_fixed
    elif method == "kate_k2":
        demo_indices = select_kate_demos(tgt_sem, src_sem, k=2)
        demo_getter = lambda i: [source_train[j] for j in demo_indices[i]]
    elif method == "jac_k2":
        demo_indices = select_jac_demos(target_test, source_train, k=2)
        demo_getter = lambda i: [source_train[j] for j in demo_indices[i]]
    elif method == "cider_k2":
        # Reuse the earlier baseline's CIDER pipeline (with seed for NB fold)
        candidate_idx, _ = select_candidate_source(source_train, src_sem, tgt_sem,
                                                     h=H_CANDIDATES, gamma=GAMMA)
        candidate_pairs = [source_train[j] for j in candidate_idx]
        candidate_sem = src_sem[candidate_idx]
        candidate_struct = structural_vectors_for_pairs(candidate_pairs)
        target_struct = structural_vectors_for_pairs(target_test)
        sim_matrix = cider_similarity_matrix(tgt_sem, candidate_sem,
                                              target_struct, candidate_struct, alpha=ALPHA)
        candidate_labels = [p["label"] for p in candidate_pairs]
        demo_indices = select_demos_for_targets(sim_matrix, k=K_DEMOS,
                                                  candidate_labels=candidate_labels)
        demo_getter = lambda i: [candidate_pairs[j] for j in demo_indices[i]]
    elif method == "zero_shot":
        demo_getter = lambda i: []

    else:
        raise ValueError(method)

    # LLM inference
    client, model = CLIENTS[backbone_name]
    preds, labels = [], []
    total_prompt_tokens = 0
    total_completion_tokens = 0
    total_api_latency_sec = 0.0
    for i, target_pair in enumerate(target_test):
        demos = demo_getter(i)
        prompt = build_cider_prompt(target_pair, demos, src_dom, tgt_dom)
        resp, pt, ct, lat = llm_call_with(client, model, prompt)
        preds.append(parse_yes_no(resp))
        labels.append(target_pair["label"])
        total_prompt_tokens += pt
        total_completion_tokens += ct
        total_api_latency_sec += lat
        if (i + 1) % 500 == 0:
            f1t, _, _ = compute_metrics(preds, labels)
            print(f"    [{i+1}/{n_test}] running F1={f1t:.4f}")

    f1, p, r = compute_metrics(preds, labels)
    elapsed = time.time() - t_start
    print(f"  [done] F1={f1:.4f}, P={p:.4f}, R={r:.4f}, elapsed={elapsed:.0f}s")

    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / "predictions.jsonl", "w") as f:
        for i, (pd, lb) in enumerate(zip(preds, labels)):
            f.write(json.dumps({"i": i, "pred": pd, "label": lb}) + "\n")

    payload = {
        "method": method,
        "backbone": backbone_name,
        "model": model,
        "source_dataset": source_ds,
        "target_dataset": target_ds,
        "seed": seed,
        "n_test_pairs": n_test,
        "positive_rate": pos_rate,
        "elapsed_sec": elapsed,
        "api_latency_sec": total_api_latency_sec,
        "prompt_tokens_total": total_prompt_tokens,
        "completion_tokens_total": total_completion_tokens,
        "prompt_tokens_mean": total_prompt_tokens / max(n_test, 1),
        "completion_tokens_mean": total_completion_tokens / max(n_test, 1),
        "test_f1": f1,
        "test_precision": p,
        "test_recall": r,
        "returncode": 0,
    }
    (out_dir / "metrics.json").write_text(json.dumps(payload, indent=2))
    print(f"  [wrote] {out_dir}/metrics.json")
    return payload

## 8. Main matrix loop

In [ ]:
# ---------------------------------------------------------------------------
# Budget cap: halts the matrix run if cumulative cost exceeds MAX_BUDGET_USD.
# Uses DeepInfra 2026-Q3 pricing (in USD per million tokens). Update as needed.
# ---------------------------------------------------------------------------
MAX_BUDGET_USD = 20.0

PRICING_USD_PER_MTOKEN = {
    # DeepInfra
    "meta-llama/Llama-3.3-70B-Instruct":         (0.23, 0.40),
    "meta-llama/Meta-Llama-3.1-8B-Instruct":     (0.03, 0.06),
    "Qwen/Qwen2.5-72B-Instruct":                 (0.35, 0.40),
    # Together (fallback identifiers, if any backbone entry uses Together)
    "meta-llama/Llama-3.3-70B-Instruct-Turbo":   (0.88, 0.88),
    "meta-llama/Llama-3.1-8B-Instruct-Turbo":    (0.18, 0.18),
    "Qwen/Qwen2.5-72B-Instruct-Turbo":           (1.20, 1.20),
}


def _cell_cost_usd(payload):
    if not payload:
        return 0.0
    price = PRICING_USD_PER_MTOKEN.get(payload.get("model"))
    if not price:
        return 0.0
    in_price, out_price = price
    return (payload.get("prompt_tokens_total", 0) * in_price
            + payload.get("completion_tokens_total", 0) * out_price) / 1_000_000


all_results = []
running_cost_usd = 0.0
budget_hit = False

for backbone_name in RUN_CONFIG["backbones"]:
    if budget_hit:
        break
    for (source_ds, target_ds) in RUN_CONFIG["pairs"]:
        if budget_hit:
            break
        for method in RUN_CONFIG["methods"]:
            if budget_hit:
                break
            for seed in SEEDS:
                if running_cost_usd > MAX_BUDGET_USD:
                    print(f"\n[BUDGET CAP] cumulative ${running_cost_usd:.2f} > "
                          f"${MAX_BUDGET_USD:.2f} — halting further cells.")
                    budget_hit = True
                    break
                try:
                    r = run_one_cell(backbone_name, source_ds, target_ds, method, seed)
                    all_results.append(r)
                    running_cost_usd += _cell_cost_usd(r)
                    print(f"  [$ running total = ${running_cost_usd:.2f} / ${MAX_BUDGET_USD}]")
                except Exception as e:
                    print(f"[ERR] {backbone_name}/{target_ds}/{method}/seed_{seed}: {e}")
                    import traceback; traceback.print_exc()

print(f"\n[complete] SUBSET={SUBSET}, cells run: {len(all_results)}, "
      f"total cost ${running_cost_usd:.2f}")
if budget_hit:
    print("(halted at budget cap; increase MAX_BUDGET_USD and rerun to continue —"
          " completed cells are skipped via SKIP_IF_DONE)")

## 9. Summary

In [ ]:
print(f"{'backbone':<15} {'target':<35} {'method':<12} {'seed':>5} {'F1':>8}")
print("-" * 80)
for r in all_results:
    if "test_f1" in r:
        print(f"{r['backbone']:<15} {r['target_dataset']:<35} {r['method']:<12} "
              f"{r['seed']:>5} {r['test_f1']:>8.4f}")

## 10. Push results from Colab

```bash
cd $REPO_ROOT
git add results/runs/main_matrix/
git status
git commit -m "main matrix main paper matrix run: SUBSET=phase1"
git push
```

Then in `03_statistical_analysis.ipynb` on Mac, aggregate + compute Wilcoxon.